# Actividad: Redes Neuronales II
## 02. Red Multicapa (MLP) desde Cero con Backpropagation y ReLU/Sigmoide
Implementación analítica de la regla de la cadena para descenso de gradiente.


In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

class MLPScratch:
    def __init__(self, layers=[2, 16, 8, 1], act='relu', lr=0.1, epochs=1000):
        self.layers, self.act, self.lr, self.epochs = layers, act, lr, epochs
        self.params = {}
        for l in range(1, len(layers)):
            scale = np.sqrt(2.0/layers[l-1]) if act == 'relu' else np.sqrt(1.0/layers[l-1])
            self.params[f'W{l}'] = np.random.randn(layers[l], layers[l-1]) * scale
            self.params[f'b{l}'] = np.zeros((layers[l], 1))

    def _act(self, z):
        return np.maximum(0, z) if self.act == 'relu' else 1/(1+np.exp(-np.clip(z, -250, 250)))

    def _d_act(self, a):
        return (a > 0).astype(float) if self.act == 'relu' else a*(1-a)

    def forward(self, X):
        cache = {'A0': X.T}
        L = len(self.layers) - 1
        for l in range(1, L):
            Z = np.dot(self.params[f'W{l}'], cache[f'A{l-1}']) + self.params[f'b{l}']
            cache[f'A{l}'] = self._act(Z)
        ZL = np.dot(self.params[f'W{L}'], cache[f'A{L-1}']) + self.params[f'b{L}']
        cache[f'A{L}'] = 1.0 / (1.0 + np.exp(-np.clip(ZL, -250, 250)))
        return cache[f'A{L}'], cache

    def backward(self, Y, cache):
        grads = {}
        m = Y.shape[1]
        L = len(self.layers) - 1
        dZ = cache[f'A{L}'] - Y
        grads[f'dW{L}'] = (1.0/m) * np.dot(dZ, cache[f'A{L-1}'].T)
        grads[f'db{L}'] = (1.0/m) * np.sum(dZ, axis=1, keepdims=True)
        for l in reversed(range(1, L)):
            dA_prev = np.dot(self.params[f'W{l+1}'].T, dZ)
            dZ = dA_prev * self._d_act(cache[f'A{l}'])
            grads[f'dW{l}'] = (1.0/m) * np.dot(dZ, cache[f'A{l-1}'].T)
            grads[f'db{l}'] = (1.0/m) * np.sum(dZ, axis=1, keepdims=True)
        return grads

    def fit(self, X, y):
        Y = y.reshape(1, -1)
        L = len(self.layers) - 1
        for epoch in range(self.epochs):
            AL, cache = self.forward(X)
            grads = self.backward(Y, cache)
            for l in range(1, L+1):
                self.params[f'W{l}'] -= self.lr * grads[f'dW{l}']
                self.params[f'b{l}'] -= self.lr * grads[f'db{l}']
        return self

    def predict(self, X):
        AL, _ = self.forward(X)
        return (AL >= 0.5).astype(int).flatten()

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

mlp = MLPScratch(act='relu', lr=0.1, epochs=1000).fit(X_train, y_train)
acc = accuracy_score(y_test, mlp.predict(X_test))
print(f"Exactitud MLP ReLU Scratch: {acc*100:.2f}%")
